# Files and Tables

DuckDB reads and writes CSV, Parquet and JSON files itself. This tutorial imports a CSV file as a table, saves a Julia table into the database, joins the two and exports the result.

It needs DuckDB.jl, DBInterface.jl and Tables.jl. It uses an **in-memory** database and a temporary folder, so nothing is left behind.

In [ ]:
Pollis.packages("DuckDB", "DBInterface", "Tables")

In [ ]:
# DuckDB is built into Pollis: use its environment if this project does not have DuckDB
pollis_env = joinpath(homedir(), ".pollis", "julia", "environments", "v$(VERSION.major).$(VERSION.minor)")
isdir(pollis_env) && !(pollis_env in LOAD_PATH) && push!(LOAD_PATH, pollis_env)

using DuckDB, DBInterface, Tables, Statistics, Dates

cd(mktempdir())
con = DBInterface.connect(DuckDB.DB, ":memory:")

## A CSV File

A small file with the awkward details of real exports: semicolons, decimal commas, day-first dates and `NA` for missing values.

In [ ]:
open("sales.csv", "w") do io
	println(io, "id;region;day;price")
	for i in 1:200
		price = i % 17 == 0 ? "NA" : replace(string(round(10 + (i * 37 % 900) / 10, digits=2)), "." => ",")
		println(io, i, ";", ("north", "south", "east", "west")[1 + i % 4], ";", lpad(1 + i % 28, 2, '0'), "/0", 1 + i % 9, "/2026;", price)
	end
end
print(join(first(readlines("sales.csv"), 4), "\n"))

## Import It

`read_csv` detects the delimiter and header. The decimal comma, the date format and the missing marker are given explicitly. `DESCRIBE` checks the column types before the table is created.

In [ ]:
reader = "read_csv('sales.csv', decimal_separator = ',', dateformat = '%d/%m/%Y', nullstr = 'NA')"
Tables.columntable(DBInterface.execute(con, "DESCRIBE SELECT * FROM $reader"))

In [ ]:
DBInterface.execute(con, "CREATE OR REPLACE TABLE sales AS SELECT * FROM $reader")
Tables.columntable(DBInterface.execute(con, "SELECT count(*) AS rows, count(price) AS prices, min(day) AS first_day FROM sales"))

A file can also be queried without importing it. That is handy for a quick look; a table is faster when the file is queried again and again.

In [ ]:
Tables.columntable(DBInterface.execute(con, "SELECT region, count(*) AS n FROM $reader GROUP BY region ORDER BY region"))

## Save a Julia Table

`DuckDB.register_table` shows a Julia table to SQL as a view, without copying it. `CREATE TABLE ... AS SELECT` then stores a copy in the database, and the view is removed.

In [ ]:
targets = (region = ["north", "south", "east", "west"], target = [2500.0, 2000.0, 2200.0, 1800.0])
DuckDB.register_table(con, targets, "julia_data")
DBInterface.execute(con, "CREATE OR REPLACE TABLE targets AS SELECT * FROM julia_data")
DuckDB.unregister_table(con, "julia_data")
Tables.columntable(DBInterface.execute(con, "SELECT table_name, estimated_size, column_count FROM duckdb_tables() ORDER BY table_name"))

## Combine Them

Imported and saved tables are ordinary tables: a join compares sales with their targets.

In [ ]:
report = Tables.columntable(DBInterface.execute(con, """
SELECT region, round(sum(price), 2) AS sales, target,
       round(100 * sum(price) / target, 1) AS pct_of_target
FROM sales JOIN targets USING (region)
GROUP BY region, target
ORDER BY pct_of_target DESC
"""))

## Export

`COPY (query) TO 'file'` writes a result straight to a file. Parquet keeps the column types; CSV is readable anywhere.

In [ ]:
DBInterface.execute(con, "COPY (SELECT * FROM sales) TO 'sales.parquet' (FORMAT parquet)")
DBInterface.execute(con, "COPY (SELECT * FROM sales) TO 'sales_clean.csv' (FORMAT csv, HEADER)")
filesize("sales.csv"), filesize("sales_clean.csv"), filesize("sales.parquet")

Reading the Parquet file back gives exactly the same columns, types included. That is the round trip a CSV file cannot promise.

In [ ]:
original = Tables.columntable(DBInterface.execute(con, "SELECT * FROM sales ORDER BY id"))
back = Tables.columntable(DBInterface.execute(con, "SELECT * FROM read_parquet('sales.parquet') ORDER BY id"))
[(name, eltype(back[name]), isequal(original[name], back[name])) for name in keys(original)]

## Append

`INSERT INTO ... BY NAME` adds rows by matching column names, so the new data need not list its columns in the same order.

In [ ]:
DuckDB.register_table(con, (price = [99.5], id = [201], region = ["north"], day = [Date(2026, 12, 31)]), "julia_data")
DBInterface.execute(con, "INSERT INTO sales BY NAME SELECT * FROM julia_data")
DuckDB.unregister_table(con, "julia_data")
Tables.columntable(DBInterface.execute(con, "SELECT * FROM sales ORDER BY id DESC LIMIT 2"))

In [ ]:
DBInterface.close!(con)